In [17]:
# import packages
from bs4 import BeautifulSoup
import requests

In [18]:
# assign the URL of the page to be crawled & request the page  
url = 'https://en.wikipedia.org/wiki/List_of_largest_companies_in_the_United_States_by_revenue'
page = requests.get(url)
soup = BeautifulSoup(page.text, 'html.parser')

In [19]:
# find the first table on the page
table = soup.find_all('table', {'class':'wikitable'})[0]

In [20]:
# get the name of the columns in the table
for i in table.find_all('th'):
    print(i.text)

Rank

Name

Industry

Revenue (USD millions)

Revenue growth

Employees

Headquarters



In [21]:
# assign the column names to a list
titles = [header.text.strip() for header in table.find_all('th')]

In [22]:
print(titles)

['Rank', 'Name', 'Industry', 'Revenue (USD millions)', 'Revenue growth', 'Employees', 'Headquarters']


In [23]:
# import pandas
import pandas as pd

In [ ]:
# iterate through all the rows and extract the data, clean and append to the DataFrame
rows = []
for i in table.find_all('tr')[1:]:
    row_data = []  
    for j in i.find_all('td'):
        row_data.append(j.text.strip())
    rows.append(row_data)

# create a DataFrame from the list of rows
df = pd.DataFrame(rows, columns=titles)

[['1', 'Walmart', 'Retail', '648,125', '6.0%', '2,100,000', 'Bentonville, Arkansas'], ['2', 'Amazon', 'Retail and cloud computing', '574,785', '11.9%', '1,525,000', 'Seattle, Washington'], ['3', 'Apple', 'Electronics industry', '383,482', '-2.8%', '161,000', 'Cupertino, California'], ['4', 'UnitedHealth Group', 'Healthcare', '371,622', '14.6%', '440,000', 'Minnetonka, Minnesota'], ['5', 'Berkshire Hathaway', 'Conglomerate', '364,482', '20.7%', '396,500', 'Omaha, Nebraska'], ['6', 'CVS Health', 'Healthcare', '357,776', '10.9%', '259,500', 'Woonsocket, Rhode Island'], ['7', 'ExxonMobil', 'Petroleum industry', '344,582', '-16.7%', '61,500', 'Spring, Texas'], ['8', 'Alphabet', 'Technology and cloud computing', '307,394', '8.7%', '182,502', 'Mountain View, California'], ['9', 'McKesson Corporation', 'Health', '276,711', '4.8%', '48,000', 'Irving, Texas'], ['10', 'Cencora', 'Pharmacy wholesale', '262,173', '9.9%', '44,000', 'Conshohocken, Pennsylvania'], ['11', 'Costco', 'Retail', '242,290',

In [25]:
df.head(10)
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Rank                    100 non-null    object
 1   Name                    100 non-null    object
 2   Industry                100 non-null    object
 3   Revenue (USD millions)  100 non-null    object
 4   Revenue growth          100 non-null    object
 5   Employees               100 non-null    object
 6   Headquarters            100 non-null    object
dtypes: object(7)
memory usage: 5.6+ KB


,Rank,Name,Industry,Revenue (USD millions),Revenue growth,Employees,Headquarters
count,100,100,100,100,100,100,100
unique,100,100,37,98,96,98,71
top,1,Walmart,Financials,"54,317",4.8%,"226,000","New York City, New York"
freq,1,1,13,2,2,2,13


In [26]:
# split the Headquaters column into city and state for better representation

df.iloc[:, :] = df.iloc[:, :].astype({'Rank': int, 'Name': str, 'Industry': str, 'Revenue (USD millions)': str, 'Revenue growth': str, 'Employees': str, 'Headquarters': str})
df['Revenue (USD millions)'] = df['Revenue (USD millions)'].str.replace(',', '').astype(float).div(1000).round(2)     
df['Revenue growth'] = df['Revenue growth'].str.replace('%', '').astype(float)  
df['Employees'] = df['Employees'].str.replace(',', '').astype(int)


In [27]:
# clean the new columns
df[['city', 'state']] = df['Headquarters'].str.rsplit(',',n= 1, expand=True)  
df['state'] = df['state'].str.strip()
df['city'] = df['city'].str.strip()

In [28]:
# make the column names lower case and replace spaces with underscores
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [29]:
# rename the revenue column
df.rename(columns={"revenue_(usd_millions)" : "revenue(usd_billions)", "revenue_growth" : "revenue_growth(%)"}, inplace=True)   

In [30]:
# drop the original headquarters column
df.drop('headquarters', axis=1, inplace=True)

In [31]:
# save the DataFrame to a CSV file for visualization
df.to_csv(r"C:\Users\joysn\Desktop\biggest_companies.csv", index=False)